In [2]:
import yt_dlp
from langchain_huggingface import HuggingFaceEmbeddings
from sklearn.metrics.pairwise import cosine_similarity
from langchain_text_splitters import RecursiveCharacterTextSplitter


class YTVideoFetcher:
    def __init__(self, topic, embedding_model, k=5):
        self.embedding_model = embedding_model
        
        self.topic = topic
        
        self.k = k
        self.imp_params = ['title', 'id', 'description', 'duration', 'view_count', 'like_count', 'webpage_url']
        self.url = f"ytsearch{k}:{self.topic}"
        self.ydl = yt_dlp.YoutubeDL({
            "queit":True
        })
        self.results = []
        self.search_videos()
        self.metadata = self.extract_metadata()

        

    def search_videos(self):

        self.results = self.ydl.extract_info(
            self.url,
            download=False
        )
    
    def extract_metadata(self):

        metadata = []

        for video in self.results["entries"]:

            video_data = {}

            for param in self.imp_params:
                video_data[param] = video.get(param)

            metadata.append(video_data)

        return metadata
    
    def filter_docs(self):
        topic_embedding = self.embedding_model.embed_query(
            self.topic
        )

        texts = [
            f"{doc['title']} {doc['description'][:500]}"
            for doc in self.metadata
        ]

        doc_embeddings = self.embedding_model.embed_documents(
            texts
        )

        for doc, emb in zip(self.metadata, doc_embeddings):
            doc["semantic_score"] = cosine_similarity(
                [topic_embedding],
                [emb]
            )[0][0]

        self.metadata.sort(
            key=lambda x: x["semantic_score"],
            reverse=True
        )

        self.metadata = self.metadata[: max(1, self.k // 2)]

        self.metadata.sort(
            key=lambda x: (
                x.get("view_count", 0),
                x.get("like_count", 0) or 0
            ),
            reverse=True
        )

        return self.metadata

In [3]:
ytf = YTVideoFetcher("docker", k=2, embedding_model=HuggingFaceEmbeddings(model="all-MiniLM-l6-v2"))
ytf.metadata

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4880.13it/s]


[youtube:search] Extracting URL: ytsearch2:docker
[download] Downloading playlist: docker
[youtube:search] query "docker": Downloading web client config
[youtube:search] query "docker" page 1: Downloading API JSON
[youtube:search] Playlist docker: Downloading 2 items of 2
[download] Downloading item 1 of 2
[youtube] Extracting URL: https://www.youtube.com/watch?v=3c-iBn73dDE
[youtube] 3c-iBn73dDE: Downloading webpage


[youtube] 3c-iBn73dDE: Downloading android vr player API JSON
[download] Downloading item 2 of 2
[youtube] Extracting URL: https://www.youtube.com/watch?v=9bSbNNH4Nqw
[youtube] 9bSbNNH4Nqw: Downloading webpage


[youtube] 9bSbNNH4Nqw: Downloading android vr player API JSON
[download] Finished downloading playlist: docker


[{'title': 'Docker Tutorial for Beginners [FULL COURSE in 3 Hours]',
  'id': '3c-iBn73dDE',
  'description': '►  Grab your free DevOps Roadmap: https://bit.ly/3GE9mzp\nFull Docker Tutorial | Complete Docker Course | Hands-on course with a lot of demos and explaining the concepts behind, so that you really understand it.\n\n💚    Become a DevOps Engineer - full educational program:  https://bit.ly/3WvLq53\n💙    Become a Kubernetes Administrator - CKA:                       https://bit.ly/3WwgLF5 \n\n►  Follow me on IG for behind the scenes content:                     👉🏼   https://bit.ly/2F3LXYJ\n\n#docker #dockertutorial #techworldwithnana\n\nBy the end, you will have a deep understanding of the concepts and a great overall big picture of how Docker is used in the whole software development process. \nThe course is a mix of animated theoretic explanation and hands-on demo’s to follow along, so you get your first hands-on experience with Docker and feel more confident using it in your pr

In [4]:
from youtube_transcript_api import YouTubeTranscriptApi

ytt_api = YouTubeTranscriptApi()

response = ytt_api.fetch('3c-iBn73dDE')

In [5]:
len(response.snippets)

3533

In [6]:
def chunk_transcript(
    transcript,
    max_chars=3000
):
    chunks = []

    current_text = []
    start_time = None
    current_len = 0

    for seg in transcript:
        text = seg.text

        if start_time is None:
            start_time = seg.start

        if current_len + len(text) > max_chars:

            end_time = seg.start

            chunks.append({
                "content": " ".join(current_text),
                "start_time": start_time,
                "end_time": end_time
            })

            current_text = []
            start_time = seg.start
            
            current_len = 0

        current_text.append(text)
        current_len += len(text)

    return chunks

In [7]:
type(response.snippets[0])

youtube_transcript_api._transcripts.FetchedTranscriptSnippet

In [8]:
chunks = chunk_transcript(response.snippets, max_chars=1000)

In [9]:
len(chunks)

126

In [10]:
embedding_model = HuggingFaceEmbeddings(model="all-MiniLM-l6-v2")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8064.76it/s]


In [11]:
from langchain_classic.schema import Document

docs = [Document(page_content=chunk['content'], metadatas={"start_time": chunk['start_time'], "end_time":chunk['end_time']}) for chunk in chunks]



In [12]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(docs, embedding_model)

C:\Users\tejas\AppData\Local\Temp\ipykernel_26248\2918311453.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [14]:
dimension = vector_store.index.d
num_vectors = vector_store.index.ntotal

size_bytes = dimension * num_vectors * 4
print(size_bytes / (1024**2), "MB")

0.1845703125 MB


In [15]:
retriever = vector_store.as_retriever(
    search_kwargs={"k":5}
)

In [16]:
docs = retriever.invoke(
    "How to download docker"
)

for doc in docs:
    print(doc.page_content)

enable you to execute some Docker commands Docker compose if you don't know it yet don't worry about it but it's just technology to orchestrate if you have multiple containers um and some other stuff that we're not going to need in this tutorial but you will have everything in a package installed so go ahead and download the stable version well I already have Docker installed from The Edge channel so I won't be installing it again but it shouldn't matter because the steps of installation are the same for both so once the docker DMG file is downloaded you just double click on it and it will pop up this window just drag the docker whale app into the applications and it will be installed on your Mach as the next step you will see Docker installed in your applications so you can just go ahead and start it so as you see the docker sign or icon is starting here if you click on it you see the status that Docker is running and you can configure some preferences and check the docker version
you

In [17]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile"
)

query = "what is explained in the video"

docs = retriever.invoke(query)

context = "\n\n".join(
    doc.page_content
    for doc in docs
)

prompt = f"""
Answer the question using ONLY the context below.

Context:
{context}

Question:
{query}
"""

response = llm.invoke(prompt)

print(response.content)

The video explains the concepts of Docker, including what Docker is, what problems it solves, and how it works on an operating system level. It also covers the difference between Docker and virtual machines, such as Oracle Virtual Box. Additionally, the video will go through a complete workflow with a demo project, including developing locally with containers, running multiple containers with Docker Compose, building a Docker image, and pushing it to a private repository. The video also touches on persisting data in Docker and configuring persistence for a demo project.


In [18]:
from pydantic import BaseModel, Field
class schema(BaseModel):
    subtopics: list[str] = Field(
        description="A list of subtopics related to user's topic, each topic shoud be concise and suitable as a section in study notes."
    )



In [19]:
structured_llm = llm.with_structured_output(schema)


In [20]:
subtopics = structured_llm.invoke("docker").subtopics

In [ ]:
from langchain_classic.agents import create_tool_calling_agent

In [22]:
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import WikipediaQueryRun
import wikipedia
wrapper = WikipediaAPIWrapper(
    top_k_results=5,            
    doc_content_chars_max=100, 
    lang="en"                   
)

tool = WikipediaQueryRun(api_wrapper=wrapper)


for i in subtopics:

    results = wikipedia.search(i)
    print(results)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [23]:
seen_titles = set()


for subtopic in subtopics:
    results = wikipedia.search(subtopic)

    for result in results:
        page = wikipedia.page(result)

        if page.title not in seen_titles:
            seen_titles.add(page.title)
            break

JSONDecodeError: Expecting value: line 1 column 1 (char 0)